In [2]:
import os
import random
from utils.hand_model_lite import HandModelMJCFLite
import numpy as np
import transforms3d
import torch
import trimesh


/opt/conda/envs/dexgraspnet/lib/python3.7/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# import numpy as np

# # Заменить 'имя_файла.npy' на путь к твоему файлу
# data = np.load('../data/dataset/hummer.npy', allow_pickle=True)

# print(data)

In [3]:
mesh_path = "../data/meshdata"
# hand_name ="shadow_dexee"
# hand_name ="barret"
# hand_name ="DIP-Flex_opened_kinematics"
# hand_name = "robotiq_2"
# hand_name = "panda"
hand_name ="hand_camera"

use_visual_mesh = True

if hand_name =="shadow_dexee":
    '''For shadow dexee'''
    data_path = "../data/dataset/shadow_dexee/"
    hand_file = "mjcf/shadow_dexee.xml"
    joint_names = [
                    "F0_J0", "F0_J1", "F0_J2", "F0_J3", "F1_J0", "F1_J1", "F1_J2", "F1_J3", "F2_J0", "F2_J1", "F2_J2", "F2_J3"
    ]

elif hand_name =="barret":
    ''' For BarretHand'''
    data_path = "../data/dataset/barret/"
    hand_file = "mjcf/barret.xml"
    joint_names = [
                    "wam_bhand_finger_1_prox_joint", "wam_bhand_finger_1_med_joint", "wam_bhand_finger_1_dist_joint", 
                    "wam_bhand_finger_2_prox_joint", "wam_bhand_finger_2_med_joint", "wam_bhand_finger_2_dist_joint",
                    "wam_bhand_finger_3_med_joint", "wam_bhand_finger_3_dist_joint"
    ]

elif hand_name =="DIP-Flex_opened_kinematics":
    ''' For Egorhand'''
    data_path = "../data/dataset/DIP-Flex_opened_kinematics"
    hand_file = "mjcf/DIP-Flex_opened_kinematics.xml"
    joint_names = [
                    "Joint_pinkie_abduction", "Joint_pinkie_PPflexion", "Joint_pinkie_DPflexion",
                    "Joint_index_abduction", "Joint_index_PPflexion", "Joint_index_DPflexion",
                    "Joint_thumb_rotation", "Joint_thumb_abduction", "Joint_thumb_PPflexion", "Joint_thumb_DPflexion"
    ]

elif hand_name == "robotiq_2":
    '''For robotiq'''
    data_path = "../data/dataset/robotiq_2/"
    hand_file = "mjcf/robotiq_2 simpl.xml"
    joint_names = [
                    "left_spring_link_joint", "left_follower",
                    "right_spring_link_joint", "right_follower_joint"
    ]

elif hand_name == "panda":
    '''For panda'''
    data_path = "../data/dataset/panda/"
    hand_file = "mjcf/panda.xml"
    joint_names = [
                    "finger_joint1", "finger_joint2"
    ]

elif hand_name == "allegro":
    '''For allegro hand'''
    data_path = "../data/dataset/allegro/"
    hand_file = "mjcf/allegro.xml"
    joint_names = [
                    "ffj0", "ffj1", "ffj2", "ffj3",
                                "mfj0", "mfj1", "mfj2", "mfj3",
                                "rfj0", "rfj1", "rfj2", "rfj3",
                                "thj0", "thj1", "thj2", "thj3"
    ]

elif hand_name == "hand_camera":
    '''For Olga's hand with camera'''
    data_path = "../data/dataset/hand_camera/"
    hand_file = "mjcf/hand_camera.xml"
    joint_names = [
                    "joint_proximal_thumb", "joint_mideal_thumb", "joint_distal_thumb",
                    "joint_mideal_finger_first", "joint_distal_finger_first",
                    "joint_mideal_finger_second", "joint_distal_finger_second"
    ]

translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']


In [4]:
hand_model = HandModelMJCFLite(
    hand_file,
    "mjcf/assets/" + hand_name)

In [33]:
grasp_code_list = []
for code in os.listdir(data_path):
    grasp_code_list.append(code[:-4])

print(grasp_code_list)

['sem-Bottle-437678d4bc6be981c8724d5673a063a6', 'screwdriver_0', 'screwdriver_5', 'hummer_0', 'core-mug-8570d9a8d24cb0acbebd3c0c0c70fb03', 'sem-Camera-7bff4fd4dc53de7496dece3f86cb5dd5']


In [47]:
grasp_code = random.choice(grasp_code_list)
grasp_data = np.load(
    os.path.join(data_path, grasp_code+".npy"), allow_pickle=True)
object_mesh_origin = trimesh.load(os.path.join(
    mesh_path, grasp_code, "coacd/decomposed.obj"))
print(grasp_code)

# print(grasp_data)
print(len(grasp_data))

sem-Bottle-437678d4bc6be981c8724d5673a063a6
100


In [55]:
import random
import numpy as np
import torch
import transforms3d
import trimesh

# Добавляем импорт для работы с файлами
import os
import json

# Файл для сохранения состояния
STATE_FILE = 'grasp_index_state.json'

# Загружаем сохраненный индекс или устанавливаем 0
if os.path.exists(STATE_FILE):
    with open(STATE_FILE, 'r') as f:
        state = json.load(f)
        index = state.get('current_index', 0)
else:
    index = 0

print(f"Текущий индекс: {index}")

# Проверяем, не превышает ли индекс доступное количество поз
if index >= len(grasp_data):
    print("Все позы были просмотрены. Сбрасываю счетчик.")
    index = 0
    

qpos = grasp_data[index]['qpos']
rot = np.array(transforms3d.euler.euler2mat(
    *[qpos[name] for name in rot_names]))
rot = rot[:, :2].T.ravel().tolist()
hand_pose = torch.tensor([qpos[name] for name in translation_names] + rot + [qpos[name]
                         for name in joint_names], dtype=torch.float, device="cpu").unsqueeze(0)
hand_model.set_parameters(hand_pose)
hand_mesh = hand_model.get_trimesh_data(0)
object_mesh = object_mesh_origin.copy().apply_scale(grasp_data[index]["scale"])

# Задаем цвета
hand_color = [0.7, 0.7, 0.7, 1.0]
object_color = [0.2, 0.5, 0.8, 1.0]

# Применяем цвета к мешам
hand_mesh.visual.face_colors = hand_color
object_mesh.visual.face_colors = object_color

(hand_mesh+object_mesh).show()

# Увеличиваем индекс для следующего запуска
index += 1

# Сохраняем новый индекс в файл
with open(STATE_FILE, 'w') as f:
    json.dump({'current_index': index}, f)

print(f"Следующая поза будет иметь индекс: {index}")

(hand_mesh+object_mesh).show()


Текущий индекс: 57
Следующая поза будет иметь индекс: 58
